In [51]:
import pandas as pd
import numpy as np
import json
import os
from glob import glob

In [52]:
tasks = ['bbh', 'gsm8k', 'humaneval_instruct', 'ifeval', 'squadv2']
tasks

['bbh', 'gsm8k', 'humaneval_instruct', 'ifeval', 'squadv2']

In [53]:
base_paths = glob('eval-results/*/*/*/*.csv')
skip_paths = glob('eval-results/*/*/*/skip*.csv')
all_paths = base_paths + skip_paths

In [54]:
for task in tasks:
	task_paths = [path for path in all_paths if f'/{task}/' in path]
	# Merge all CSV files for the task
	dfs = []
	for path in task_paths:
		df = pd.read_csv(path)
		df['model'] = path.split('/')[2]  # Extract model name from path
		df['task_general'] = path.split('/')[3]  # Extract task name from path
		df['setting'] = path.split('/')[4].replace('.csv', '')  # Extract setting (base/skip) from path
		dfs.append(df)
	merged_df = pd.concat(dfs, ignore_index=True)
	# Save the merged DataFrame to a new CSV file
	merged_df.to_csv(f'merged_{task}.csv', index=False)

In [55]:
task2score_column = {
	'bbh': 'exact_match,get-answer',
	'gsm8k': "exact_match,flexible-extract",
    'humaneval_instruct': "pass@1,create_test",
    "squadv2": "f1",
    "ifeval": "prompt_level_loose_acc",
}

In [56]:
for task in tasks:
	task_df = pd.read_csv(f'merged_{task}.csv')
	models = list(task_df['model'].unique())
	score_column = task2score_column[task]
	subtasks = list(task_df['task'].unique())
	results = []
	for model in models:
		for subtask in subtasks:
			base_score = task_df[(task_df['model'] == model) & (task_df['task'] == subtask) & (task_df['setting'] == 'base')][score_column].values[0]
			skip_rows = task_df[(task_df['model'] == model) & (task_df['task'] == subtask) & (task_df['setting'].str.startswith('skip'))]
			skip_rows['diff_score'] = skip_rows[score_column] - base_score
			skip_rows['diff_score_percent'] = skip_rows['diff_score'] / base_score * 100
			skip_rows_max = skip_rows.loc[skip_rows['diff_score'].idxmax()]
			skip_value_max = skip_rows_max[score_column]
			skip_diff_max = skip_rows_max['diff_score']
			skip_diff_percent_max = skip_rows_max['diff_score_percent']
			skip_setting_max = skip_rows_max['setting']
			num_rows_positive_diff = (skip_rows['diff_score'] > 0).sum()
			if task == "squadv2":
				base_score_str = f"{base_score:.2f}%"
				skip_diff_max_str = f"{skip_diff_max:.2f}%"
			else:
				base_score_str = f"{base_score*100:.2f}%"
				skip_diff_max_str = f"{skip_diff_max*100:.2f}%"
			results.append({
				'model': model,
				'subtask': subtask,
				'skip_setting_max': skip_setting_max,
				'skip_value_max': skip_value_max,
				'skip_diff_percent_max': skip_diff_percent_max,
				'base_score': base_score_str,
				'skip_layer_max': skip_setting_max.split('_')[-1],
				'skip_diff_max': skip_diff_max_str,
				'num_rows_positive_diff': num_rows_positive_diff
			})
	results_df = pd.DataFrame(results)
	results_df.to_csv(f'{task}_analysis_results.csv', index=False)

In [ ]:
# # Get 4 values for each subtask in bbh:'base' setting score, diff between 'base' and 'skip*' setting
# results = []
# for model in models:
# 	for subtask in subtasks:
# 		base_score = bbh_df[(bbh_df['model'] == model) & (bbh_df['task'] == subtask) & (bbh_df['setting'] == 'base')][score_column].values[0]
# 		skip_rows = bbh_df[(bbh_df['model'] == model) & (bbh_df['task'] == subtask) & (bbh_df['setting'].str.startswith('skip'))]
# 		skip_rows['diff_score'] = skip_rows[score_column] - base_score
# 		skip_rows['diff_score_percent'] = skip_rows['diff_score'] / base_score * 100
# 		skip_rows_max = skip_rows.loc[skip_rows['diff_score'].idxmax()]
# 		skip_value_max = skip_rows_max[score_column]
# 		skip_diff_max = skip_rows_max['diff_score']
# 		skip_setting_max = skip_rows_max['setting']
# 		num_rows_positive_diff = (skip_rows['diff_score'] > 0).sum()
# 		results.append({
# 			'model': model,
# 			'subtask': subtask,
# 			'base_score': base_score,
# 			'skip_setting_max': skip_setting_max,
# 			'skip_value_max': skip_value_max,
# 			'skip_diff_max': skip_diff_max,
# 			'num_rows_positive_diff': num_rows_positive_diff
# 		})

In [40]:
results_df = pd.DataFrame(results)
results_df.to_csv('bbh_analysis_results.csv', index=False)